# Sesión 0.8: Carga e Ingesta de Archivos Delimitados con Python Nativo (Ingesta 'From Scratch')
## Diplomado en Ciencia de Datos e Inteligencia Artificial - INT210

---

### 1. La Analogía Infantil: El Archivero con Llave de Seguridad y las Fichas de Papel

Imagina una oficina con un gran archivero de metal:

- **La sentencia `with open()`:** El empleado usa su llave para abrir la gaveta, saca las fichas que necesita para trabajar, y en cuanto termina (o incluso si ocurre una emergencia), la gaveta se cierra automáticamente con llave. Así evitamos dejar expedientes abiertos que bloqueen el paso o agoten los recursos del sistema (*File Descriptor Leaks*).
- **El lector `csv.reader`:** Es una lupa que lee cada ficha línea por línea, separando los datos donde encuentra comas y entregándotelos limpios y ordenados.

En Machine Learning 'From Scratch', aprender a ingerir y tipar datos sin librerías externas nos otorga un control absoluto sobre el consumo de memoria y la velocidad de procesamiento.

---
### 2. Rigor Científico: Streams de I/O, Context Managers y Tipado Nativo (Joel Grus, Cap. 9)

La manipulación de archivos a bajo nivel se rige por el protocolo de **Administradores de Contexto** (*Context Managers*):

1. **Seguridad RAII (*Resource Acquisition Is Initialization*):** `with open(...) as f:` ejecuta `f.__enter__()` al inicio y garantiza la ejecución de `f.__exit__()` al salir del bloque, liberando el descriptor de archivo en el núcleo del sistema operativo.
2. **Carga en Flujo (*Streaming*):** `csv.reader` no carga el archivo completo en memoria RAM, sino que lee y decodifica línea a línea bajo demanda mediante el protocolo iterador.
3. **Estructuras Tipadas con `namedtuple`:** Una `namedtuple` proporciona acceso por nombre de campo (`t.monto`) con la misma inmutabilidad y bajo consumo de memoria que una tupla estándar en CPython.

In [ ]:
import csv
from collections import namedtuple
from typing import List

# Definición de una estructura de registro inmutable mediante namedtuple
Transaccion = namedtuple(
    'Transaccion',
    ['id_transaccion', 'usuario', 'monto_usd', 'ip_origen', 'pais_origen', 'hora_registro', 'etiqueta_real']
)

def cargar_transacciones_csv(ruta_archivo: str) -> List[Transaccion]:
    """
    Lee un archivo CSV mediante streaming nativo y retorna una lista de NamedTuples tipadas.
    """
    transacciones = []
    with open(ruta_archivo, mode='r', encoding='utf-8', newline='') as archivo_csv:
        lector = csv.DictReader(archivo_csv)
        for fila in lector:
            registro = Transaccion(
                id_transaccion=fila['id_transaccion'],
                usuario=fila['usuario'],
                monto_usd=float(fila['monto_usd']),  # Casteo explícito a float
                ip_origen=fila['ip_origen'],
                pais_origen=fila['pais_origen'],
                hora_registro=fila['hora_registro'],
                etiqueta_real=fila['etiqueta_real']
            )
            transacciones.append(registro)
    return transacciones

# Carga de datos reales sin dependencias de Pandas
dataset_transacciones = cargar_transacciones_csv('transacciones_seguridad.csv')

print(f"=== Total de Registros Ingeridos 'From Scratch': {len(dataset_transacciones)} ===")
for t in dataset_transacciones[:3]:
    print(f"ID: {t.id_transaccion} | Usuario: {t.usuario:12s} | Monto: ${t.monto_usd:8.2f} | País: {t.pais_origen}")

#### Explicación Técnica Línea por Línea del Bloque de Código:

- `Transaccion = namedtuple(...)`: Crea una subclase de tupla con campos accesibles por atributo, consumiendo significativamente menos memoria que un diccionario estándar de Python.
- `with open(ruta_archivo, mode='r', encoding='utf-8', newline='') as archivo_csv`: Abre el flujo de lectura especificando codificación universal UTF-8 y manejo consistente de saltos de línea multi-plataforma.
- `csv.DictReader(archivo_csv)`: Mapea automáticamente la primera fila del CSV como claves de un diccionario para cada registro subsecuente.
- `monto_usd=float(fila['monto_usd'])`: Castea la cadena de texto leída del disco a un número de coma flotante nativo para habilitar cálculos aritméticos.
- `transacciones.append(registro)`: Almacena el objeto estructurado inmutable en la lista en memoria principal.

---
### 3. Caso de Estudio: Motor Heurístico de Detección de Fraude 'From Scratch'

Aplicamos un clasificador por reglas directamente sobre las `NamedTuples` sin usar librerías de Machine Learning.

**Regla de Fraude:**
Una transacción se clasifica como `'Fraude'` si el monto es $\ge 5000.0$ USD **Y** el país de origen no es `'Bolivia'`.

In [ ]:
def clasificador_fraude_nativo(t: Transaccion) -> str:
    if t.monto_usd >= 5000.0 and t.pais_origen != 'Bolivia':
        return 'Fraude'
    else:
        return 'Normal'

# Evaluación y cálculo de métricas 'from scratch'
resultados = []
for t in dataset_transacciones:
    pred = clasificador_fraude_nativo(t)
    resultados.append({'id': t.id_transaccion, 'monto': t.monto_usd, 'real': t.etiqueta_real, 'pred': pred})

print("=== Auditoría de Fraude Financiero ===")
for r in resultados:
    print(f"ID: {r['id']} | Monto: ${r['monto']:8.2f} | Real: {r['real']:7s} | Predicho: {r['pred']}")

#### Explicación Técnica Línea por Línea:
- `t.monto_usd >= 5000.0 and t.pais_origen != 'Bolivia'`: Evaluación de predicados lógicos con acceso por campo de la `NamedTuple`.
- `for t in dataset_transacciones:`: Iteración nativa sobre la lista estructurada con tiempo de ejecución lineal $\mathcal{O}(N)$.

---
### 4. Ejercicio Práctico Guiado: Agregación Estadística 'From Scratch' (`# TODO`)

**Instrucción:** Completa la función `calcular_monto_promedio_por_pais` para que retorne un diccionario con el monto promedio de transacciones por cada país sin usar Pandas.

In [ ]:
from collections import defaultdict

def calcular_monto_promedio_por_pais(transacciones: List[Transaccion]) -> dict:
    """
    Calcula la media aritmética del monto en USD agrupada por país de origen.
    """
    ### TU CÓDIGO AQUÍ
    pass

# Comprobación del ejercicio
# promedios = calcular_monto_promedio_por_pais(dataset_transacciones)
# print(f"Montos Promedio por País: {promedios}")